# Silver quality és feltöltés

Ez a notebook a bronze OHLCV és silver/calendar adatokból előállítja a silver generated OHLCV táblát.

A folyamat:

1. Betölti a bronze és calendar adatokat Azure-ból.
2. Calendar alapján szűri az elvárt gyertyahelyeket.
3. Generált OHLC gyertyát készít.
4. Quality mezőket ad hozzá:
   - `consensus_quality`
   - `candle_quality`
   - `is_outlier`
   - `quality_status`
5. Broker ranking táblát készít.
6. Feltölti a silver adatokat Azure-ba havi és asset bontásban.

Két generálási módszer van:

- `method=0`: medián alapú OHLC
- `method=1`: automatikus preferred broker alapú OHLC

A két módszer külön Azure útvonalra kerül:

- `silver/generated_ohlcv/method_0/...`
- `silver/generated_ohlcv/method_1/...`


In [7]:
START_MONTH = "2024-01"
END_MONTH = "2025-12"
BROKERS = None
ASSETS = None
INTERVAL = "1m"

In [8]:
import importlib

import quality.data_loader
import quality.calendar_alignment
import quality.candle_generation

importlib.reload(quality.data_loader)
importlib.reload(quality.calendar_alignment)
importlib.reload(quality.candle_generation)

from quality.data_loader import load_bronze_ohlcv
from quality.calendar_alignment import load_calendar
from quality.candle_generation import build_broker_wide_table


bronze_df = load_bronze_ohlcv(
    start_month=START_MONTH,
    end_month=END_MONTH,
    brokers=BROKERS,
    assets=ASSETS,
    interval=INTERVAL,
    source="azure",
    print_progress=True,
)

calendar_df = load_calendar(
    start_month=START_MONTH,
    end_month=END_MONTH,
    assets=ASSETS,
    interval=INTERVAL,
    source="azure",
    print_progress=True,
)

quality_wide_df = build_broker_wide_table(
    bronze_df=bronze_df,
    calendar_df=calendar_df,
)

quality_wide_df.shape

reading bronze/binance/btcusd/2024/01/BTCUSDT-1m-2024-01.parquet
reading bronze/binance/btcusd/2024/02/BTCUSDT-1m-2024-02.parquet
reading bronze/binance/btcusd/2024/03/BTCUSDT-1m-2024-03.parquet
reading bronze/binance/btcusd/2024/04/BTCUSDT-1m-2024-04.parquet
reading bronze/binance/btcusd/2024/05/BTCUSDT-1m-2024-05.parquet
reading bronze/binance/btcusd/2024/06/BTCUSDT-1m-2024-06.parquet
reading bronze/binance/btcusd/2024/07/BTCUSDT-1m-2024-07.parquet
reading bronze/binance/btcusd/2024/08/BTCUSDT-1m-2024-08.parquet
reading bronze/binance/btcusd/2024/09/BTCUSDT-1m-2024-09.parquet
reading bronze/binance/btcusd/2024/10/BTCUSDT-1m-2024-10.parquet
reading bronze/binance/btcusd/2024/11/BTCUSDT-1m-2024-11.parquet
reading bronze/binance/btcusd/2024/12/BTCUSDT-1m-2024-12.parquet
reading bronze/binance/btcusd/2025/01/BTCUSDT-1m-2025-01.parquet
reading bronze/binance/btcusd/2025/02/BTCUSDT-1m-2025-02.parquet
reading bronze/binance/btcusd/2025/03/BTCUSDT-1m-2025-03.parquet
reading bronze/binance/bt

(4191720, 36)

In [9]:
import quality.silver_ohlcv

importlib.reload(quality.silver_ohlcv)
importlib.reload(quality.candle_generation)

from quality.candle_generation import generate_candles_from_brokers
from quality.silver_ohlcv import (
    build_silver_ohlcv_from_generated,
    build_silver_ohlcv_summary,
)


method_0_generated_df = generate_candles_from_brokers(
    quality_wide_df,
    method=0,
)

method_0_silver_ohlcv_df = build_silver_ohlcv_from_generated(
    method_0_generated_df,
)

method_0_summary_df = build_silver_ohlcv_summary(
    method_0_silver_ohlcv_df,
)

method_0_summary_df

,asset,consensus_quality,candle_quality,is_outlier,quality_status,rows,generated_rows,start,end,avg_broker_count,max_close_diff_pct,max_abs_return_pct
0,BTCUSD,multi_source,bad,False,bad,230,230,2024-01-05 01:47:00+00:00,2025-10-10 23:16:00+00:00,2.000000,3.365631,1.795249
1,BTCUSD,multi_source,bad,True,bad,4,4,2024-01-05 01:48:00+00:00,2025-10-10 21:21:00+00:00,2.000000,5.355351,2.588428
2,BTCUSD,multi_source,good,False,good,981420,981420,2024-01-01 21:12:00+00:00,2025-12-31 14:58:00+00:00,2.000000,0.100000,1.861671
3,BTCUSD,multi_source,good,True,bad,5,5,2024-01-03 12:01:00+00:00,2025-01-23 20:12:00+00:00,2.000000,0.076114,2.885488
4,BTCUSD,multi_source,warning,False,warning,60188,60188,2024-01-02 00:15:00+00:00,2025-12-31 14:54:00+00:00,2.000000,0.299918,1.949372
5,BTCUSD,multi_source,warning,True,bad,3,3,2024-08-05 01:10:00+00:00,2024-12-05 22:28:00+00:00,2.000000,0.202889,3.672727
6,BTCUSD,single_source,warning,False,warning,10790,10790,2024-01-01 00:00:00+00:00,2025-12-31 23:59:00+00:00,1.000000,0.000000,1.303564
7,DAX,missing,missing,False,missing,21,0,2024-01-24 08:00:00+00:00,2025-10-02 07:01:00+00:00,NaN,NaN,NaN
8,DAX,multi_source,bad,False,bad,469,469,2024-03-21 08:00:00+00:00,2025-12-30 16:39:00+00:00,2.000000,2.614862,0.667693
9,DAX,multi_source,bad,True,bad,10,10,2024-04-05 07:00:00+00:00,2025-04-09 11:00:00+00:00,2.000000,1.804817,1.432111


In [10]:
method_1_generated_df = generate_candles_from_brokers(
    quality_wide_df,
    method=1,
)

method_1_silver_ohlcv_df = build_silver_ohlcv_from_generated(
    method_1_generated_df,
)

method_1_summary_df = build_silver_ohlcv_summary(
    method_1_silver_ohlcv_df,
)

method_1_summary_df


,asset,consensus_quality,candle_quality,is_outlier,quality_status,rows,generated_rows,start,end,avg_broker_count,max_close_diff_pct,max_abs_return_pct
0,BTCUSD,multi_source,bad,False,bad,229,229,2024-01-05 01:47:00+00:00,2025-10-10 23:16:00+00:00,2.000000,3.365631,1.887407
1,BTCUSD,multi_source,bad,True,bad,5,5,2024-01-05 01:48:00+00:00,2025-10-10 21:21:00+00:00,2.000000,5.355351,4.289804
2,BTCUSD,multi_source,good,False,good,981420,981420,2024-01-01 21:12:00+00:00,2025-12-31 14:58:00+00:00,2.000000,0.100000,1.815083
3,BTCUSD,multi_source,good,True,bad,5,5,2024-01-03 12:01:00+00:00,2025-01-23 20:12:00+00:00,2.000000,0.076114,2.908606
4,BTCUSD,multi_source,warning,False,warning,60188,60188,2024-01-02 00:15:00+00:00,2025-12-31 14:54:00+00:00,2.000000,0.299918,1.901710
5,BTCUSD,multi_source,warning,True,bad,3,3,2024-08-05 01:10:00+00:00,2024-12-05 22:28:00+00:00,2.000000,0.202889,3.675565
6,BTCUSD,single_source,warning,False,warning,10790,10790,2024-01-01 00:00:00+00:00,2025-12-31 23:59:00+00:00,1.000000,0.000000,1.303564
7,DAX,missing,missing,False,missing,21,0,2024-01-24 08:00:00+00:00,2025-10-02 07:01:00+00:00,NaN,NaN,NaN
8,DAX,multi_source,bad,False,bad,472,472,2024-03-21 08:00:00+00:00,2025-12-30 16:39:00+00:00,2.000000,1.543310,0.715070
9,DAX,multi_source,bad,True,bad,7,7,2024-08-05 07:06:00+00:00,2025-04-07 15:14:00+00:00,2.000000,2.614862,2.053114


In [11]:
import quality.broker_ranking

importlib.reload(quality.broker_ranking)

from quality.broker_ranking import build_broker_ranking


method_0_broker_ranking_df = build_broker_ranking(
    method_0_generated_df,
)

method_1_broker_ranking_df = build_broker_ranking(
    method_1_generated_df,
)

method_0_broker_ranking_df


,asset,broker,calendar_rows,broker_rows,missing_rows,coverage_ratio,mean_abs_diff,median_abs_diff,max_abs_diff,mean_abs_diff_pct,median_abs_diff_pct,max_abs_diff_pct
0,BTCUSD,binance,1052640,1052640,0,1.000000,15.192886,10.23500,2784.120000,0.017654,0.012858,2.607846
1,BTCUSD,dukascopy,1052640,1041850,10790,0.989750,15.350232,10.40000,2784.120000,0.017837,0.013055,2.607846
2,DAX,interactive_brokers,263640,262879,761,0.997113,1.138452,0.77950,242.183250,0.005446,0.003789,1.290558
3,DAX,dukascopy,263640,259627,4013,0.984778,1.152712,0.79175,242.183250,0.005515,0.003848,1.290558
4,EURUSD,saxo_bank,738990,716928,22062,0.970146,0.000017,0.00001,0.000785,0.001547,0.000970,0.067273
5,EURUSD,dukascopy,738990,701686,37304,0.949520,0.000004,0.00000,0.000708,0.000333,0.000000,0.061985
6,EURUSD,interactive_brokers,738990,662175,76815,0.896054,0.000003,0.00000,0.000485,0.000311,0.000000,0.043316
7,US500,interactive_brokers,692970,652337,40633,0.941364,1.549878,1.42225,32.379750,0.026365,0.024930,0.491289
8,US500,dukascopy,692970,651299,41671,0.939866,1.552348,1.42425,32.379750,0.026407,0.024974,0.491289
9,XAGUSD,saxo_bank,721740,707767,13973,0.980640,0.001548,0.00060,0.404000,0.004336,0.001953,0.747817


In [12]:
import importlib

import pipelines.silver_generated_ohlcv_runner

importlib.reload(pipelines.silver_generated_ohlcv_runner)

from pipelines.silver_generated_ohlcv_runner import (
    upload_silver_generated_ohlcv_with_preview,
)


silver_methods = [
    {
        "generation_method_id": 0,
        "generation_method": "median_ohlc",
        "silver_ohlcv_df": method_0_silver_ohlcv_df,
    },
    {
        "generation_method_id": 1,
        "generation_method": "preferred_broker_ohlc",
        "silver_ohlcv_df": method_1_silver_ohlcv_df,
    },
]

silver_upload_results = {}

for method_config in silver_methods:
    method_id = method_config["generation_method_id"]
    method_name = method_config["generation_method"]

    print(f"Uploading silver method {method_id}: {method_name}")

    silver_upload_results[method_id] = upload_silver_generated_ohlcv_with_preview(
        silver_ohlcv_df=method_config["silver_ohlcv_df"],
        interval=INTERVAL,
        generation_method_id=method_id,
        generation_method=method_name,
        overwrite=True,
        print_progress=True,
    )

silver_upload_results[0]["upload_results"], silver_upload_results[1]["upload_results"]


Uploading silver method 0: median_ohlc
Uploading silver generated OHLCV: BTCUSD 2024-01
Uploading silver generated OHLCV: DAX 2024-01
Uploading silver generated OHLCV: EURUSD 2024-01
Uploading silver generated OHLCV: US500 2024-01
Uploading silver generated OHLCV: XAGUSD 2024-01
Uploading silver generated OHLCV: XAUUSD 2024-01
Uploading silver generated OHLCV: BTCUSD 2024-02
Uploading silver generated OHLCV: DAX 2024-02
Uploading silver generated OHLCV: EURUSD 2024-02
Uploading silver generated OHLCV: US500 2024-02
Uploading silver generated OHLCV: XAGUSD 2024-02
Uploading silver generated OHLCV: XAUUSD 2024-02
Uploading silver generated OHLCV: BTCUSD 2024-03
Uploading silver generated OHLCV: DAX 2024-03
Uploading silver generated OHLCV: EURUSD 2024-03
Uploading silver generated OHLCV: US500 2024-03
Uploading silver generated OHLCV: XAGUSD 2024-03
Uploading silver generated OHLCV: XAUUSD 2024-03
Uploading silver generated OHLCV: BTCUSD 2024-04
Uploading silver generated OHLCV: DAX 2024

(       status   asset  year  month  \
 0    uploaded  BTCUSD  2024      1   
 1    uploaded     DAX  2024      1   
 2    uploaded  EURUSD  2024      1   
 3    uploaded   US500  2024      1   
 4    uploaded  XAGUSD  2024      1   
 ..        ...     ...   ...    ...   
 139  uploaded     DAX  2025     12   
 140  uploaded  EURUSD  2025     12   
 141  uploaded   US500  2025     12   
 142  uploaded  XAGUSD  2025     12   
 143  uploaded  XAUUSD  2025     12   
 
                                            message  row_count  
 0    Silver generated OHLCV uploaded successfully.      44640  
 1    Silver generated OHLCV uploaded successfully.      11440  
 2    Silver generated OHLCV uploaded successfully.      31650  
 3    Silver generated OHLCV uploaded successfully.      29040  
 4    Silver generated OHLCV uploaded successfully.      31740  
 ..                                             ...        ...  
 139  Silver generated OHLCV uploaded successfully.       9880  
 140  Silv